# Surface water mass transformation (WMT) analysis of CMIP6 output using `xwmt`

This notebook demonstrates how to compute SWMT from scratch, using only the standard ocean surface variables that are typically archived for experiments contributed to the Coupled Model Intercomparison Project Phase 6 (CMIP6) archive.

## Import packages

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import xgcm
import xwmt
import xbudget
import os

# Optional (loading to memory and plotting) 
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import calendar
import warnings

In [ ]:
print(
    'numpy version',np.__version__, '\nxarray version',xr.__version__,
    '\nxgcm version',xgcm.__version__, '\nxbudget version',xbudget.__version__,
    '\nxwmt version',xwmt.__version__,)

## Loading a CMIP6 dataset

This notebook loads GFDL-CM4 output directly from the public [Pangeo CMIP6 archive on Google Cloud Storage](https://console.cloud.google.com/storage/browser/cmip6) (anonymous, read-only access via `gcsfs`), so it can be run by anyone without access to GFDL's internal filesystem. This is the **same published ESGF/CMIP6 data** that GFDL also serves locally, so for identical facets and time period the values are bit-for-bit identical.

All relevant variables are loaded into `ds` and the (static) grid fields are loaded into a separate list. The Pangeo zarr stores span the full 1850–2014 historical record; we subset to the last five years (Jan 2010 – Dec 2014) of the CM4 historical run via `time_slice`, matching the original `201001-201412` files. Widen `time_slice` (e.g. `slice(None)`) to use more years.

Notes on data availability for these facets on Pangeo:
- `sfdsi` (surface salt flux) is not published to the Pangeo archive, so it is skipped (it was also absent from the original GFDL-local files).
- The static `basin` and `deptho` fields are only published under the `piControl` experiment on Pangeo. Since they are time-invariant model geometry, they are identical to the historical static fields. The `basin` field (and its `flag_meanings`) is used below to build the regional mask.

In [ ]:
import gcsfs  # registers the gs:// filesystem so xarray can open the zarr stores

# Public Pangeo CMIP6 archive on Google Cloud Storage.
gcs_root = 'gs://cmip6/CMIP6'
activity_id = 'CMIP'
institution_id = 'NOAA-GFDL'
source_id = 'GFDL-CM4'
experiment_id = 'historical'
member_id = 'r1i1p1f1'
table_id = 'Omon'
grid_label = 'gn'
version = 'v20180701'
storage_options = {'token': 'anon'} # anonymous, read-only access

# Subset to the last five years (Jan 2010 - Dec 2014) to match the original
# `201001-201412` files. Set `time_slice = slice(None)` to use the full record.
time_slice = slice('2010-01-01', '2014-12-31')

def cmip6_zstore(variable_id, table_id=table_id, experiment_id=experiment_id):
    return '/'.join([
        gcs_root, activity_id, institution_id, source_id, experiment_id,
        member_id, table_id, variable_id, grid_label, version
    ]) + '/'

In [ ]:
variables = [
    'tos', # surface temperature
    'sos', # surface salinity
    'hfds', # surface heat flux
    'sfdsi', # surface salinity flux (e.g. brine rejection)
    'wfo', # surface freshwater flux
]
chunks = {'time':1, 'x':-1, 'y':-1, 'xh':-1, 'yh':-1}

ds = xr.Dataset()
for var in variables:
    try:
        da = xr.open_dataset(
            cmip6_zstore(var), engine='zarr', decode_times=True, chunks=chunks,
            backend_kwargs={'storage_options': storage_options}
        )[var]
        print('Loading', var)
        ds[var] = da.sel(time=time_slice)
    except Exception:
        print('Store for', var, 'is not available on Pangeo. Skipping.')

In [ ]:
# Static grid fields. `areacello` is published for the historical experiment, but
# `deptho` and `basin` are only published once (under piControl) on Pangeo. These are
# time-invariant model-geometry fields and are identical across experiments. The
# `basin` field (with its `flag_meanings`) is used below to build the regional mask.
grid_ds = []
static_experiment = {'areacello': experiment_id, 'deptho': 'piControl', 'basin': 'piControl'}
for var, expt in static_experiment.items():
    try:
        print('Loading', var, '(' + expt + ')')
        grid_ds.append(xr.open_dataset(
            cmip6_zstore(var, table_id='Ofx', experiment_id=expt), engine='zarr',
            chunks=chunks, backend_kwargs={'storage_options': storage_options}
        ))
    except Exception:
        print('Store for', var, 'is not available on Pangeo. Skipping.')

In [ ]:
ds = xr.merge([ds, xr.merge(grid_ds[1:], compat="override")], compat="override")

# Area needs to be loaded seperately after renaming MOM6-specific dimension names (xh, yh) 
ds['areacello'] = grid_ds[0].areacello.rename({'xh': 'x', 'yh': 'y'})

In [ ]:
# Add core coordinates of ocean_grid to ds
ds = ds.assign_coords({
    "areacello": xr.DataArray(ds["areacello"].values, dims=('y', 'x',)), # Required for area-integration
    "lon":       xr.DataArray(ds["lon"].values, dims=('y', 'x',)), # Required for calculating density if not already provided!
    "lat":       xr.DataArray(ds["lat"].values, dims=('y', 'x',)), # Required for calculating density if not already provided!
})

# xgcm grid for dataset
coords = {
    'X': {'center': 'x',},
    'Y': {'center': 'y',},
}
metrics = {
    ('X','Y'): "areacello", # Required for area-integration
}
grid = xgcm.Grid(ds, coords=coords, metrics=metrics, boundary={"X":"periodic", "Y":"extend"}, autoparse_metadata=False)

## Example plot of surface temperature

In [ ]:
ds['tos'].mean('time').plot();

## Specify mask to apply for SWMT calculation

In [ ]:
basin_name = 'pacific_tropc' # global, atlantic, indian, pacific, southern, arctic,
# atlantic_subpN,  pacific_tropc

bidx = [item.split('_')[0] for item in ds.basin.flag_meanings.split(' ')].index(basin_name.split('_')[0])

if basin_name=='global':
    mask = xr.where(ds.basin==bidx,0,1)
else:
    mask = ds.basin==bidx

In [ ]:
if basin_name[-6:]=='_tropc':
    mask = mask & (ds["lat"]<=20) & (ds["lat"]>=-20)
if basin_name[-6:]=='_subtN':
    mask = mask & (ds["lat"]<=45) & (ds["lat"]>20)
if basin_name[-6:]=='_subpN':
    mask = mask & (ds["lat"]>45)
if basin_name[-6:]=='_subtS':
    mask = mask & (ds["lat"]>=-45) & (ds["lat"]<-20)

In [ ]:
mask.plot();

## Construct xbudget dictionary

In [ ]:
budgets_dict = {
    "mass": {},
    "heat": {"surface_lambda": "tos"},
    "salt": {"surface_lambda": "sos"}
}

In [ ]:
cp = 3992.
rho_ref = 1035.
budgets_dict['heat']['rhs'] = {
    'var': None,
    'sum': {
        'var': None,
        'surface_exchange_flux_nonadvective': {
            'var': None,
            'product': {
                'var': None,
                'heat_tendency':'hfds',
                'area':'areacello'
            }
        },
        'surface_exchange_flux_advective': {
            'var': None,
            'product': {
                'var': None,
                'specific_heat_capacity': cp,
                'lambda_mass': 'tos',
                'mass_density_tendency': 'wfo',
                'area': 'areacello'
            }
        },
        'surface_ocean_flux_advective': {
            'var': None,
            'product': {
                'var': None,
                'sign': -1.,
                'specific_heat_capacity': cp,
                'lambda_mass': 'tos',
                'mass_density_tendency': 'wfo',
                'area': 'areacello'
            }
        }
    }
}
budgets_dict['salt']['rhs'] = {
    'var': None,
    'sum': {
        'var': None,
        'surface_exchange_flux_advective': {
            'var': None,
            'product': {
                'var': None,
                'unit_conversion': 0.001,
                'lambda_mass': 0.,
                'mass_density_tendency': 'wfo',
                'area': 'areacello'
            }
        },
        'surface_ocean_flux_advective': {
            'var': None,
            'product': {
                'var': None,
                'sign': -1.,
                'unit_conversion': 0.001,
                'lambda_mass': 'sos',
                'mass_density_tendency': 'wfo',
                'area': 'areacello'
            }
        }
    }
}

## Reconstruct budget terms using the 2D fields
xbudget.collect_budgets(grid._ds, budgets_dict)
simple_budget = xbudget.aggregate(budgets_dict) # aggregate to the root level

## Compute and visualize transformations

In [ ]:
# The default method using `xhistogram` for area-integrated WMT calculations but `xgcm` for spatial transformations maps
plt.figure(figsize=(10,7))
for i, method in enumerate(["default", "xgcm"]):
    print(f"Method: {method}")
    with warnings.catch_warnings():
        warnings.simplefilter(action='ignore', category=FutureWarning)
        swmt = xwmt.WaterMassTransformations(grid, simple_budget, mask=mask, cp=cp, rho_ref=rho_ref, method=method)
        G = swmt.integrate_transformations("sigma0", bins=np.arange(10, 30, 0.1))
        with ProgressBar():
            G['surface_exchange_flux_nonadvective'].load()
            G['surface_exchange_flux_advective'].load()
            G['surface_ocean_flux_advective'].load()
        
        ax = plt.subplot(2,1,i+1)
        ((G['surface_exchange_flux_nonadvective'] +
         G['surface_exchange_flux_advective'] +
         G['surface_ocean_flux_advective']
        )/rho_ref*1.e-6).T.plot(
            ax=ax,cmap='RdBu_r',vmin=-300,vmax=300,yincrease=False,
            cbar_kwargs={'label': 'Transformation (Sv)'}
        );
    plt.title(f"Vertical coordinate transformation method: {method}")
plt.tight_layout()